# Part II: Embedding variants (LSTM)

We choose the best model from Part I, the LSTM model, and explore different word embedding methods on it.  

A. randomly initialized and trainable (reused from Part I)  
B. Word2Vec Skip-gram trained on `train.txt`, frozen  
C. public GloVe: `glove.6B.100d` (Wikipedia 2014 + Gigaword 5, 6B tokens, 400K vocab, uncased. Pennington, Socher & Manning, 2014), frozen

Everything else is the same as Part I: embedding dimension 100, LSTM config, data, vocabulary, training budget (40 epochs), seed. Only the embedding table changes.

**How to run:** run Part I first (setting A is read from `results/part1/lstm.json`). GloVe is downloaded once by gensim, then run all cells in order (about 50 minutes on a CPU). Results are written to `results/part2/` and model weights to `results/models/`.

## 0. Setup

Import libraries and check hardware details. Embedding helpers are in `src/embeddings.py`.

In [1]:
import sys
from pathlib import Path
import json

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data import load_corpus, unk_rate
from src.models import RecurrentLM, set_embeddings
from src.evaluate import SEED, PROMPTS, RESULTS_DIR, Timer, save_results, hardware_summary, set_seed
from src.experiment import run_neural_model, plot_history
from src.embeddings import (
    WORD2VEC_SETTINGS, GLOVE_NAME, word2vec_sentences, train_word2vec,
    load_glove, build_embedding_matrix, nearest_neighbours,
)

set_seed(SEED)
hardware_summary()

{'platform': 'Windows-11-10.0.26200-SP0',
 'processor': 'Intel64 Family 6 Model 170 Stepping 4, GenuineIntel',
 'python': '3.13.9',
 'torch': '2.14.1+cpu',
 'torch_threads': 16,
 'device': 'cpu'}

## 1. Data

Load the same vocabulary and splits as Part I. Word2Vec gets the **train** paragraphs only (no leakage), mapped through the vocabulary so it sees exactly the tokens the LSTM sees, including `<unk>` and `<eos>`.

In [2]:
vocab, splits = load_corpus()
print(f"Vocabulary size: {len(vocab)}")

for name, ids in splits.items():
    print(f"{name} set: {len(ids)} tokens, {unk_rate(ids, vocab):.2%} unknown")

sentences = word2vec_sentences(vocab)
w2v_tokens = sum(len(sentence) for sentence in sentences)
print(f"\nWord2Vec input: {len(sentences):,} train paragraphs, {w2v_tokens:,} tokens "
      f"(same as the LM train set: {w2v_tokens == len(splits['train'])})")
print("First paragraph:", sentences[0][:15])

Vocabulary size: 4103
train set: 199118 tokens, 3.91% unknown
valid set: 12653 tokens, 5.84% unknown
test set: 9889 tokens, 4.98% unknown

Word2Vec input: 3,451 train paragraphs, 199,118 tokens (same as the LM train set: True)
First paragraph: ['my', 'father', "'s", 'family', 'name', 'being', 'pirrip', ',', 'and', 'my', 'christian', 'name', 'philip', ',', 'my']


## 2. Shared settings

Read the LSTM config and training settings from `results/part1/lstm.json`, so settings B and C match Part I exactly.

In [ ]:
part1_lstm = json.loads((RESULTS_DIR / "part1" / "lstm.json").read_text(encoding="utf-8"))
RECURRENT_CONFIG = part1_lstm["config"]
TRAINING = part1_lstm["training"]
VOCAB_SIZE = len(vocab)

print("LSTM config:", RECURRENT_CONFIG)
print("Training:", TRAINING)

## 3. Setting A: random trainable embeddings (reused from Part I)

The Part I LSTM already has the same dimension, architecture, data and training, so it is reused instead of retrained.

In [5]:
print("Setting A: random trainable embeddings (Part I LSTM)")
print(f"Trainable parameters: {part1_lstm['trainable_parameters']:,}")
print(f"Best epoch: {part1_lstm['best_epoch']} (valid perplexity {part1_lstm['valid_perplexity']:.2f})")
print(f"Test NLL: {part1_lstm['test_nll']:.4f}")
print(f"Test perplexity: {part1_lstm['test_perplexity']:.2f}")
print(f"Training time: {part1_lstm['train_seconds']:.0f} s")

Setting A: random trainable embeddings (Part I LSTM)
Trainable parameters: 1,831,363
Best epoch: 30 (valid perplexity 76.26)
Test NLL: 4.2913
Test perplexity: 73.06
Training time: 1321 s


## 4. Setting B: own Word2Vec embeddings

**Train Word2Vec:** Skip-gram with gensim, adapted from the Lecture 2 call (`sg=1`, `window=5`, `negative=5`, `epochs=15`, `workers=1`, `seed=4213`) with `vector_size=100` to match the LSTM. The time is the embedding-training cost, reported separately from LM training.

In [6]:
print("Word2Vec settings:", WORD2VEC_SETTINGS)

with Timer() as w2v_timer:
    word2vec = train_word2vec(sentences)

print(f"Trained Word2Vec in {w2v_timer.seconds:.1f} s: {len(word2vec.wv):,} vectors of size {word2vec.wv.vector_size}")

Word2Vec settings: {'vector_size': 100, 'window': 5, 'min_count': 1, 'sg': 1, 'negative': 5, 'ns_exponent': 0.75, 'sample': 0.0, 'shrink_windows': False, 'epochs': 15, 'workers': 1, 'seed': 4213}
Trained Word2Vec in 13.4 s: 4,102 vectors of size 100


**Build the embedding table:** one 100-d row per vocabulary id. `<pad>` is zeros, any token without a vector gets random values with the same spread as the found vectors (seed 4213). The same rule is used for GloVe.

In [9]:
def show_coverage(report):
    print(f"Vectors found: {report['found']:,} / {report['vocabulary_size']:,} vocabulary tokens "
          f"({report['type_coverage']:.1%})")
    print(f"Train-token coverage: {report['train_token_coverage']:.1%}")
    print(f"Missing ({report['missing']}): {report['missing_examples']}")
    print(f"Missing rows initialised as {report['missing_init']}")
    print(f"Mean vector norm: {report['mean_vector_norm']:.2f}")

w2v_matrix, w2v_report = build_embedding_matrix(vocab, word2vec.wv, train_ids=splits["train"])
show_coverage(w2v_report)

Vectors found: 4,102 / 4,102 vocabulary tokens (100.0%)
Train-token coverage: 100.0%
Missing (0): []
Missing rows initialised as normal(0, 0.2362) with seed 4213; <pad> = zeros
Mean vector norm: 2.30


## 5. Setting C: public GloVe embeddings

**Load GloVe:** GloVe 6B 100d (Pennington et al., 2014), loaded as `glove-wiki-gigaword-100` via `gensim.downloader`. It was trained on ~6B tokens of Wikipedia and Gigaword news, far more data than our ~200k training tokens. The first run downloads it.

In [25]:
with Timer() as glove_timer:
    glove = load_glove()

print(f"Loaded {GLOVE_NAME} in {glove_timer.seconds:.1f} s: {len(glove):,} words, dimension {glove.vector_size}")

Loaded glove-wiki-gigaword-100 in 36.7 s: 400,000 words, dimension 100


**Build the embedding table:** same procedure as Word2Vec.

In [11]:
glove_matrix, glove_report = build_embedding_matrix(vocab, glove, train_ids=splits["train"])
show_coverage(glove_report)

Vectors found: 4,046 / 4,102 vocabulary tokens (98.6%)
Train-token coverage: 93.8%
Missing (56): ['<unk>', '<eos>', 'wemmick', 'pumblechook', 'wopsle', 'drummle', 'orlick', 'compeyson', 'trabb', 'gargery', 'skiffins', 'startop', 'bargemen', 'flopson', 'meantersay', 'halloa', "look'ee", 'waldengarver', 'jaggerth', 'oncommon']
Missing rows initialised as normal(0, 0.5183) with seed 4213; <pad> = zeros
Mean vector norm: 5.14


**Nearest neighbours:** the closest vocabulary words (cosine similarity) to a few words from the novel, in each table. This shows what each embedding encodes before LM training.

In [12]:
QUERY_WORDS = ["joe", "sister", "convict", "money", "gentleman", "dark"]

for word in QUERY_WORDS:
    print(word)
    for name, matrix in [("Word2Vec", w2v_matrix), ("GloVe", glove_matrix)]:
        print(f"  {name:<8}: {', '.join(nearest_neighbours(word, matrix, vocab, k=6))}")

joe
  Word2Vec: rob, tickler, learning, carols, disgrace, n
  GloVe   : tom, mike, frank, jack, john, baker
sister
  Word2Vec: echoed, sternly, deliberate, ram, son, grandpapa
  GloVe   : daughter, mother, wife, niece, aunt, cousin
convict
  Word2Vec: nevvy, wolf, causes, wise, daily, stopping
  GloVe   : convicts, acquitted, murderer, murder, slave, conviction
money
  Word2Vec: spend, compensation, prepared, paid, finish, colonel
  GloVe   : cash, paying, pay, paid, get, keep
gentleman
  Word2Vec: young, pale, man, angel, fortun, common
  GloVe   : servant, gentlemen, polite, idiot, thief, man
dark
  Word2Vec: crowded, wicket, police, loitering, evenings, summer
  GloVe   : bright, black, darker, light, shadows, blue


## 6. LSTM with frozen Word2Vec

Same as the Part I LSTM, with one extra step: copy in the Word2Vec table with `set_embeddings(..., freeze=True)`. The seed is set before building, so every other layer starts from the same weights as in Part I. Trainable parameters drop from 1,831,363 to 1,421,063 (the 410,300 embedding weights are frozen).

In [ ]:
set_seed(SEED)
w2v_model = RecurrentLM(VOCAB_SIZE, cell="lstm", **RECURRENT_CONFIG)
set_embeddings(w2v_model, w2v_matrix, freeze=True)
w2v_results = run_neural_model("part2/lstm_word2vec", w2v_model, splits, vocab, TRAINING,
                               config={**RECURRENT_CONFIG, "embeddings": "own Word2Vec, frozen"},
                               description="LSTM + frozen Word2Vec")

for prompt, text in w2v_results["samples"].items():
    print(f"\n[{prompt}]\n{text}")

In [ ]:
plot_history(w2v_results["history"], "LSTM + frozen Word2Vec: training and validation loss", "part2/lstm_word2vec_loss.png")
plt.show()

## 7. LSTM with frozen GloVe

Identical to section 6, with the GloVe table.

In [ ]:
set_seed(SEED)
glove_model = RecurrentLM(VOCAB_SIZE, cell="lstm", **RECURRENT_CONFIG)
set_embeddings(glove_model, glove_matrix, freeze=True)
glove_results = run_neural_model("part2/lstm_glove", glove_model, splits, vocab, TRAINING,
                                 config={**RECURRENT_CONFIG, "embeddings": "GloVe 6B 100d, frozen"},
                                 description="LSTM + frozen GloVe")

for prompt, text in glove_results["samples"].items():
    print(f"\n[{prompt}]\n{text}")

In [ ]:
plot_history(glove_results["history"], "LSTM + frozen GloVe: training and validation loss", "part2/lstm_glove_loss.png")
plt.show()

**Save:** model weights to `results/models/` (reused in Part III), and the embedding details (settings, cost, coverage) to `results/part2/embeddings.json`.

In [ ]:
MODELS_DIR = RESULTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
for name, model in [("lstm_word2vec", w2v_model), ("lstm_glove", glove_model)]:
    path = MODELS_DIR / f"{name}.pt"
    torch.save(model.state_dict(), path)
    print(f"Saved {path.name} ({path.stat().st_size / 1e6:.1f} MB)")

path = save_results("part2/embeddings", {
    "word2vec": {"settings": WORD2VEC_SETTINGS, "train_seconds": w2v_timer.seconds,
                 "train_tokens": w2v_tokens, "coverage": w2v_report},
    "glove": {"name": GLOVE_NAME, "load_seconds": glove_timer.seconds,
              "glove_vocabulary_size": len(glove), "coverage": glove_report},
    "hardware": hardware_summary(),
})
print(f"Saved to {path}")

## 8. Comparison

**Comparison tables:**  (1) prediction quality and (2) cost and coverage. Note: setting A was trained in a different session, when the laptop was busier, so its training time is not directly comparable with B and C.

In [ ]:
SETTINGS = ["lstm", "lstm_word2vec", "lstm_glove"]
LABELS = {"lstm": "A. random", "lstm_word2vec": "B. Word2Vec", "lstm_glove": "C. GloVe"}
RESULT_FILES = {"lstm": "part1/lstm", "lstm_word2vec": "part2/lstm_word2vec", "lstm_glove": "part2/lstm_glove"}
saved = {name: json.loads((RESULTS_DIR / f"{RESULT_FILES[name]}.json").read_text(encoding="utf-8"))
         for name in SETTINGS}
coverage = {"lstm": "-",
            "lstm_word2vec": f"{w2v_report['type_coverage']:.1%}",
            "lstm_glove": f"{glove_report['type_coverage']:.1%}"}
embedding_cost = {"lstm": "learned during LM training",
                  "lstm_word2vec": f"{w2v_timer.seconds:.1f} s Word2Vec on {w2v_tokens:,} tokens",
                  "lstm_glove": "pretrained on ~6B tokens (not trained here)"}

print("(1) Prediction quality")
print(f"{'setting':<12} | {'test NLL':>8} | {'test ppl':>8} | {'valid ppl':>9} | {'best epoch':>10}")
for name in SETTINGS:
    r = saved[name]
    print(f"{LABELS[name]:<12} | {r['test_nll']:>8.4f} | {r['test_perplexity']:>8.2f} | "
          f"{r['valid_perplexity']:>9.2f} | {r['best_epoch']:>10}")

print("\n(2) Cost and coverage")
print(f"{'setting':<12} | {'trainable params':>16} | {'total s':>10} | {'to best s':>9} | {'coverage':>8} | embedding cost")
for name in SETTINGS:
    r = saved[name]
    print(f"{LABELS[name]:<12} | {r['trainable_parameters']:>16,} | {r['train_seconds']:>10.0f} | "
          f"{r['seconds_to_best_epoch']:>9.0f} | {coverage[name]:>8} | {embedding_cost[name]}")

**Loss curves:** training and validation loss of the three settings, to compare convergence and overfitting.

In [ ]:
COLORS = {"lstm": "#2a78d6", "lstm_word2vec": "#eb6834", "lstm_glove": "#1baf7a"}

fig, (ax_train, ax_valid) = plt.subplots(1, 2, figsize=(11, 4))
for name in SETTINGS:
    history = saved[name]["history"]
    epochs = range(1, len(history["train_loss"]) + 1)
    best = history["best_epoch"]
    ax_train.plot(epochs, history["train_loss"], color=COLORS[name], linewidth=2, label=LABELS[name])
    ax_valid.plot(epochs, history["valid_loss"], color=COLORS[name], linewidth=2, label=LABELS[name])
    ax_valid.scatter([best], [history["valid_loss"][best - 1]], s=60, color=COLORS[name],
                     edgecolors="black", linewidths=1.2, zorder=3)

for ax, title in [(ax_train, "Training loss"), (ax_valid, "Validation loss (dot = best epoch)")]:
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "part2" / "part2_loss_curves.png", dpi=200, bbox_inches="tight")
plt.show()

**Convergence and stability:** To describe the training behaviour numerically, we calculate the first epoch within 0.05 of the best validation loss (convergence speed), the largest rise in validation loss between consecutive epochs (stability), the validation-training loss gap at the best epoch (memorising), and how far validation loss rises from the best epoch to epoch 40 (overfitting).

In [21]:
print(f"{'setting':<12} | {'near best at':>12} | {'best epoch':>10} | {'max rise':>8} | {'gap at best':>11} | {'rise to 40':>10}")
for name in SETTINGS:
    history = saved[name]["history"]
    valid, train = history["valid_loss"], history["train_loss"]
    best = history["best_epoch"]
    near_best = next(epoch for epoch, loss in enumerate(valid, 1) if loss <= valid[best - 1] + 0.05)
    max_rise = max(later - earlier for earlier, later in zip(valid, valid[1:]))
    gap = valid[best - 1] - train[best - 1]
    rise = valid[-1] - valid[best - 1]
    print(f"{LABELS[name]:<12} | {near_best:>12} | {best:>10} | {max_rise:>8.4f} | {gap:>11.4f} | {rise:>10.4f}")

setting      | near best at | best epoch | max rise | gap at best | rise to 40
A. random    |           18 |         30 |   0.0049 |      0.3485 |     0.0132
B. Word2Vec  |           24 |         39 |   0.0049 |      0.2452 |     0.0007
C. GloVe     |           24 |         40 |   0.0037 |      0.2817 |     0.0000


**Generated text:** samples for each prompt across the 3 embedding methods, with the same decoding settings as Part I.

In [22]:
for prompt in PROMPTS:
    print("=" * 100)
    print(f"PROMPT: {prompt!r}")
    for name in SETTINGS:
        print(f"\n  [{LABELS[name]}]\n  {saved[name]['samples'][prompt]}")
    print()

PROMPT: 'i was'

  [A. random]
  i was succeeded and wondered him.
tar! said estella, and in a <unk> <unk> now.
camilla, said i.
you know a moment,

  [B. Word2Vec]
  i was younger to keep up to the hummums of my guard, and a <unk> <unk> <unk> from his legs key, and the pig <unk> and <unk>, and i

  [C. GloVe]
  i was wondering of myself in the way leg. he became <unk>, and i was <unk> about one day years, and i've the going of the best of

PROMPT: 'miss havisham'

  [A. random]
  miss havisham, and so.
he has not <unk> you since, he answered at the other <unk>, and, he now the hard of human <unk>, came

  [B. Word2Vec]
  miss havisham, and all the of to morrow.
he swore, said herbert.
i was my hands, and he knew he would sit, and dry

  [C. GloVe]
  miss havisham, and then the boy was laid.
as still, said herbert.
i was his own <unk> i had seen as as ever <unk>, than

PROMPT: 'joe said'

  [A. random]
  joe said, and i had heard his <unk> to take a natural depth, and i thought for i h